In [5]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [7]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql", "sql/01_clean.sql", "sql/02_features.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

feat = load_queries("sql/02_features.sql")
display(run(feat, "features_reconcile").T)

,0
n_rows,20405666
pu_zone_null,0
pu_borough_null,0
min_hour,0
max_hour,23
min_isodow,1
max_isodow,7
n_pickup_zones,262
min_request_date,2024-12-31 00:00:00
max_request_date,2025-02-01 00:00:00


In [8]:
h = load_queries("sql/02d_hotspots.sql")
con.execute(h["create_zhw_cells"])
con.execute(h["create_zhw_hotspots"])
display(run(h, "cells_reconcile").T)

,0
n_cells,42842.0
n_zones,260.0
n_slots,168.0
max_possible_cells,43680.0
empty_cells,838.0
cells_with_no_valid_wait,8.0
n_scoped,20404892.0
n_valid,20205118.0
n_excluded,199774.0


In [9]:
display(run(h, "cell_size_distribution"))
display(run(h, "cell_size_quantiles").T)
sb = run(h, "slot_baseline")
display(sb["pct_wait_ge10"].describe().round(3).to_frame("share of waits >= 10 min across the 168 slots"))
display(sb.sort_values("pct_wait_ge10", ascending=False).head(8))

,sample_tier,n_cells,pct_of_cells,valid_rows,pct_of_valid_rows
0,1_very_small,9096,21.23,389589.0,1.93
1,2_small,28525,66.58,11900780.0,58.90
2,3_adequate,5221,12.19,7914749.0,39.17


,0
min_n,0.0
p10_n,36.0
p25_n,124.0
median_n,317.0
p75_n,655.0
p90_n,1094.9
max_n,5241.0


,share of waits >= 10 min across the 168 slots
count,168.000
mean,5.602
std,3.166
min,2.086
25%,3.414
50%,4.700
75%,6.780
max,21.149


,request_isodow,request_hour,n_scoped,n_valid,n_excluded_negative_wait,pct_excluded,n_wait_ge10,pct_wait_ge10
49,3,1,91121.0,90696.0,425.0,0.466,19181.0,21.149
50,3,2,75804.0,75439.0,365.0,0.482,13637.0,18.077
31,2,7,147393.0,143938.0,3455.0,2.344,20920.0,14.534
48,3,0,118483.0,117820.0,663.0,0.560,17082.0,14.498
4,1,4,30946.0,29200.0,1746.0,5.642,4071.0,13.942
166,7,22,105556.0,105353.0,203.0,0.192,14254.0,13.530
167,7,23,87997.0,87809.0,188.0,0.214,11476.0,13.069
79,4,7,184903.0,180835.0,4068.0,2.200,23366.0,12.921


In [10]:
display(run(h, "hotspot_summary").T)
print("Top hotspot cells (ranked by lower end of the interval)")
display(run(h, "hotspot_top_by_rate"))
print("Cells with the most long waits (impact)")
display(run(h, "hotspot_top_by_impact"))

,0
n_cells_with_valid_wait,42834.00
n_adequate_cells,5221.00
n_hotspot_1_5x,547.00
n_hotspot_2x,325.00
n_hotspot_3x,172.00
n_hotspot_robust_2x,325.00
valid_rows_in_hotspots,761943.00
long_waits_in_hotspots,176105.00
long_waits_all_cells,1051522.00
pct_of_long_waits_in_hotspots,16.75


Top hotspot cells (ranked by lower end of the interval)


,pu_borough,pu_zone,request_isodow,request_hour,n_valid,pct_excluded,avg_trips_per_day,pct_wait_ge10,pct_wait_ge10_wilson_low,pct_ge10_if_excluded_all_short,slot_pct_wait_ge10,ratio_vs_slot,zone_pct_wait_ge10_all_hours,p50_min,p90_min,pct_uber_of_valid,hotspot_robust_flag
0,Queens,JFK Airport,7,23,4011,0.000,1002.8,59.212,57.683,59.212,13.069,4.53,26.725,11.617,23.917,74.4,True
1,Queens,JFK Airport,1,0,3081,0.000,770.3,54.008,52.245,54.008,12.079,4.47,26.725,10.550,23.650,74.1,True
2,Queens,JFK Airport,7,22,4493,0.022,1123.5,50.389,48.928,50.378,13.530,3.72,26.725,10.050,17.313,75.8,True
3,Queens,JFK Airport,5,0,2949,0.068,590.2,47.202,45.405,47.170,4.582,10.30,26.725,9.633,19.273,79.1,True
4,Queens,JFK Airport,6,23,3008,0.000,752.0,47.108,45.329,47.108,8.203,5.74,26.725,9.650,16.917,78.3,True
5,Queens,JFK Airport,4,23,3893,0.026,778.8,46.185,44.624,46.174,5.870,7.87,26.725,9.367,18.670,80.0,True
6,Queens,JFK Airport,6,0,1823,0.055,456.0,46.023,43.746,45.998,6.137,7.50,26.725,9.350,18.150,77.5,True
7,Queens,LaGuardia Airport,1,21,4240,0.188,1062.0,44.811,43.320,44.727,6.762,6.63,11.948,9.167,18.435,70.6,True
8,Queens,JFK Airport,1,22,3960,0.025,990.3,44.520,42.978,44.509,7.597,5.86,26.725,9.350,17.485,77.2,True
9,Queens,JFK Airport,2,0,2427,0.082,607.3,44.705,42.737,44.669,7.724,5.79,26.725,9.317,18.440,77.2,True


Cells with the most long waits (impact)


,pu_borough,pu_zone,request_isodow,request_hour,n_valid,n_wait_ge10,pct_excluded,avg_trips_per_day,pct_wait_ge10,slot_pct_wait_ge10,ratio_vs_slot,hotspot_flag
0,Queens,JFK Airport,7,23,4011,2375,0.000,1002.8,59.212,13.069,4.53,True
1,Queens,JFK Airport,7,22,4493,2264,0.022,1123.5,50.389,13.530,3.72,True
2,Queens,LaGuardia Airport,1,21,4240,1900,0.188,1062.0,44.811,6.762,6.63,True
3,Queens,JFK Airport,4,23,3893,1798,0.026,778.8,46.185,5.870,7.87,True
4,Queens,JFK Airport,1,22,3960,1763,0.025,990.3,44.520,7.597,5.86,True
5,Queens,JFK Airport,1,0,3081,1664,0.000,770.3,54.008,12.079,4.47,True
6,Queens,JFK Airport,1,21,3961,1660,0.000,990.3,41.909,6.762,6.20,True
7,Queens,LaGuardia Airport,4,23,5241,1612,0.266,1051.0,30.757,5.870,5.24,True
8,Queens,LaGuardia Airport,7,21,3641,1525,0.165,911.8,41.884,10.070,4.16,True
9,Queens,JFK Airport,4,22,4223,1509,0.047,845.0,35.733,7.151,5.00,True


In [11]:
display(run(h, "hotspot_zone_summary"))
bs = run(h, "hotspot_by_slot")
display(bs.pivot(index="request_hour", columns="request_isodow", values="n_hotspot_cells"))
display(run(h, "hotspot_by_borough"))

,PULocationID,pu_borough,pu_zone,n_adequate_cells,n_hotspot_cells,n_robust_cells,zone_n_valid,pct_valid_rows_in_hotspots,zone_pct_wait_ge10_all_hours
0,132,Queens,JFK Airport,142,129,129,366295.0,90.88,26.725
1,138,Queens,LaGuardia Airport,119,44,44,360429.0,41.24,11.948
2,230,Manhattan,Times Sq/Theatre District,119,12,12,241197.0,16.55,7.030
3,161,Manhattan,Midtown Center,94,9,9,237099.0,15.25,5.944
4,61,Brooklyn,Crown Heights North,132,8,8,262607.0,5.24,5.554
5,17,Brooklyn,Bedford,99,8,8,170079.0,5.73,5.427
6,186,Manhattan,Penn Station/Madison Sq West,37,8,8,123590.0,8.55,4.849
7,13,Manhattan,Battery Park City,17,8,8,90323.0,11.82,4.091
8,246,Manhattan,West Chelsea/Hudson Yards,102,6,6,211001.0,6.07,4.383
9,39,Brooklyn,Canarsie,59,6,6,147417.0,5.78,7.142


request_isodow,1,2,3,4,5,6,7
request_hour,,,,,,,
0,2,2,1,1,2,3,5
1,2,1,0,1,1,3,5
2,1,0,0,0,0,2,2
3,0,0,1,0,0,0,1
4,0,0,0,0,0,0,0
5,0,0,0,1,1,0,0
6,0,0,1,2,1,1,1
7,2,2,3,7,10,1,1
8,2,3,6,9,7,1,1


,pu_borough,n_zones,n_adequate_cells,n_hotspot_cells,n_robust_cells,pct_adequate_cells_hotspot,pct_valid_rows_in_hotspots
0,Queens,69,830,180,180,21.69,11.63
1,Manhattan,67,2710,81,81,2.99,2.39
2,Brooklyn,61,1531,52,52,3.40,1.33
3,Bronx,43,150,12,12,8.00,0.57
4,Staten Island,20,0,0,0,NaN,NaN


In [12]:
out = Path("dashboard/data"); out.mkdir(parents=True, exist_ok=True)
con.execute("SELECT * FROM zhw_hotspots ORDER BY PULocationID, request_isodow, request_hour").df() \
   .to_csv(out / "2d_zone_hour_weekday_hotspots.csv", index=False)
sb.to_csv(out / "2d_slot_baseline.csv", index=False)
bs.to_csv(out / "2d_hotspot_by_slot.csv", index=False)